In [4]:
# pip install sqlalchemy

import pandas as pd
from sqlalchemy import create_engine, URL
from config import POSTGRES_CONFIG


url_conexao = URL.create(
    drivername="postgresql+psycopg2",
    username=POSTGRES_CONFIG["user"],
    password=POSTGRES_CONFIG["password"],
    host=POSTGRES_CONFIG["host"],
    port=POSTGRES_CONFIG["port"],
    database=POSTGRES_CONFIG["dbname"]
)

engine = create_engine(url_conexao)

query_silver_trajetos = """
SELECT *
FROM silver_trajetos;
"""

query_silver_embarques = """
SELECT *
FROM silver_embarques;
"""

query_silver_gastos = """
SELECT *
FROM silver_gastos;
"""

query_silver_deslocamentos = """
SELECT *
FROM silver_deslocamentos;
"""

with engine.connect() as conexao:
    df_trajetos = pd.read_sql_query(query_silver_trajetos, conexao)
    df_embarques = pd.read_sql_query(query_silver_embarques, conexao)
    df_gastos = pd.read_sql_query(query_silver_gastos, conexao)
    df_deslocamentos = pd.read_sql_query(query_silver_deslocamentos, conexao)


engine.dispose()

print("Consulta realizada com SQLAlchemy e pandas.")

Consulta realizada com SQLAlchemy e pandas.


In [6]:
import pandas as pd

ARQUIVOS = {
    "deslocamentos": {"csv": "deslocamentos.csv",    "tabela_raw": "raw_deslocamentos"},
    "gastos":        {"csv": "gastos.csv",           "tabela_raw": "raw_gastos"},
    "embarques":     {"csv": "embarques.csv",        "tabela_raw": "raw_embarques"},
    "trajetos":      {"csv": "trajetos.csv",         "tabela_raw": "raw_trajetos"},
}

df = pd.DataFrame(ARQUIVOS)

df

,deslocamentos,gastos,embarques,trajetos
csv,deslocamentos.csv,gastos.csv,embarques.csv,trajetos.csv
tabela_raw,raw_deslocamentos,raw_gastos,raw_embarques,raw_trajetos


In [7]:
a = 10
b = 0

# Realiza a divisão.
resultado = a / b

print(resultado)

ZeroDivisionError: division by zero

In [8]:
a = 10
b = 0

# Tenta executar a divisão.
try:
    resultado = a / b
    print(resultado)

# Captura o erro de divisão por zero.
except ZeroDivisionError as erro:

    # Apresenta uma mensagem personalizada.
    print(f"Não foi possível realizar a divisão. Detalhe: {erro}")

Não foi possível realizar a divisão. Detalhe: division by zero


In [11]:
# Define uma função para realizar divisões.
def dividir(a, b):

    # Tenta realizar a divisão.
    try:
        return a / b

    # Captura o erro de divisão por zero.
    except ZeroDivisionError as erro:

        # Lança uma nova exceção com uma mensagem personalizada.
        raise RuntimeError(
            f"Não foi possível dividir {a} por {b}. "
            f"Detalhe: {erro}"
        )


# Executa a função.
dividir(10, 0)

RuntimeError: Não foi possível dividir 10 por 0. Detalhe: division by zero

In [ ]:
-- Cria uma view agregada com informações de três tabelas.
CREATE OR REPLACE VIEW vw_resumo_viagens_integrado AS

SELECT
    d.nome_orgao_superior AS orgao,
    COUNT(*) AS total_viagens,
    SUM(d.valor_total) AS custo_total,
    ROUND(AVG(d.valor_total), 2) AS custo_medio,
    COALESCE(SUM(g.total_pago), 0) AS total_pagamentos,
    COALESCE(SUM(t.total_trechos), 0) AS total_trechos

-- Define a tabela principal.
FROM silver_deslocamentos AS d

-- Relaciona os gastos previamente agrupados por viagem.
LEFT JOIN (
    SELECT
        id_viagem,
        SUM(valor) AS total_pago
    FROM silver_gastos
    GROUP BY id_viagem
) AS g ON d.id_viagem = g.id_viagem

-- Relaciona os trechos previamente agrupados por viagem.
LEFT JOIN (
    SELECT
        id_viagem,
        COUNT(*) AS total_trechos
    FROM silver_trajetos
    GROUP BY id_viagem
) AS t ON d.id_viagem = t.id_viagem

-- Consolida as informações por órgão superior.
GROUP BY d.nome_orgao_superior;

-- Consulta os indicadores consolidados por órgão.
SELECT *
FROM vw_resumo_viagens_integrado
ORDER BY custo_total DESC NULLS LAST;

,id_viagem,num_proposta,situacao,viagem_urgente,cod_orgao_superior,nome_orgao_superior,cod_orgao_solicitante,nome_orgao_solicitante,nome_viajante,cargo,data_inicio,data_fim,destinos,motivo,valor_diarias,valor_passagens,valor_devolucao,valor_outros_gastos,valor_total,duracao_dias
0,20547601,000050/25,Realizada,SIM,52000,Ministério da Defesa,52000,Ministério da Defesa - Unidades com vínculo di...,JAILTON BARBOSA GOMES,,2025-01-07,2025-01-08,Recife/PE,Desmobilização do 19º Contingente Do Exército ...,0.00,2051.66,0.0,0.0,2051.66,1
1,20547602,000051/25,Realizada,NÃO,52000,Ministério da Defesa,52000,Ministério da Defesa - Unidades com vínculo di...,JOAO VITOR LEITE ALVES,,2025-01-07,2025-01-08,Recife/PE,Desmobilização do 19º Contingente Do Exército ...,0.00,2051.66,0.0,0.0,2051.66,1
2,20600997,000006/25,Realizada,NÃO,52000,Ministério da Defesa,52111,Comando da Aeronáutica,JOHNATA TAVARES SOARES,,2025-01-15,2025-01-18,Rio de Janeiro/RJ,Realizar treinamento para torna-se Instrutor d...,0.00,1275.63,0.0,0.0,1275.63,3
3,20557377,000080/25,Realizada,NÃO,52000,Ministério da Defesa,52000,Ministério da Defesa - Unidades com vínculo di...,LUCAS JAFFE DE CARVALHO GOIS,,2025-01-08,2025-01-08,Fortaleza/CE,Desmobilização do 19° contingente do EB.,0.00,1677.76,0.0,0.0,1677.76,0
4,20290601,000001/25,Realizada,NÃO,26000,Ministério da Educação,26237,Universidade Federal de Juiz de Fora,GRIGORI CHAPIRO,PROFESSOR DO MAGISTERIO SUPERIOR,2025-01-16,2025-01-31,Palo Alto - Califórnia/Estados Unidos da América,"Colaboração científica na Stanford University,...",0.00,0.00,0.0,0.0,0.00,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,20616206,000012/25,Realizada,NÃO,26000,Ministério da Educação,26234,Universidade Federal do Espírito Santo,MARIA LUISA RAMOS DE OLIVEIRA SOARES,,2025-01-12,2025-01-17,Vitória/ES,Membro de banca examinadora do concurso public...,0.00,932.86,0.0,0.0,932.86,5
996,20616575,000001/25-1C,Realizada,SIM,26000,Ministério da Educação,26402,"Instituto Federal de Educação, Ciência e Tecno...",RENATO DE MEI ROMERO,PROFESSOR ENS BASICO TECN TECNOLOGICO,2025-01-25,2025-02-01,Palmas/TO,Participação em evento científico XXV Encontro...,2622.75,2904.11,0.0,0.0,5526.86,7
997,20616874,000022/25-1C,Realizada,SIM,52000,Ministério da Defesa,52111,Comando da Aeronáutica,GREGORY BENJAMIM MEDEIROS VALADARES,,2025-01-05,2025-01-10,Itaituba/PA,"REALIZAR MANUTENÇÃO CORRETIVA, CONFORME MCI DE...",0.00,1304.09,0.0,0.0,1304.09,5
998,20617005,000253/25,Não realizada,NÃO,52000,Ministério da Defesa,52121,Comando do Exército,AMERICO PAYSAN VALDETARO FILHO,,2025-01-09,2025-01-10,Rio de Janeiro/RJ,"Centro de custos F039925S, PASSAGEM DE COMANDO...",0.00,1794.93,0.0,0.0,1794.93,1


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, URL
from config import POSTGRES_CONFIG


url_conexao = URL.create(
    drivername="postgresql+psycopg2",
    username=POSTGRES_CONFIG["user"],
    password=POSTGRES_CONFIG["password"],
    host=POSTGRES_CONFIG["host"],
    port=POSTGRES_CONFIG["port"],
    database=POSTGRES_CONFIG["dbname"]
)

engine = create_engine(url_conexao)

query = """
SELECT *
FROM vw_gold_resumo_orgao;

"""

with engine.connect() as conexao:
    df_view = pd.read_sql_query(query, conexao)    


engine.dispose()

#EXIBIR RESULTADO
display(df_view.info())
print(50*'*')
display(df_view.head())

print("Consulta realizada com SQLAlchemy e pandas.")